# DataMate · GPU inference

Chọn Colab, Kaggle hoặc GPU thuê trong ô đầu; sửa đường dẫn theo dữ liệu đã upload. Notebook dùng chung application.run_batch với CLI/Docker, không khóa benchmark. Chuẩn bị checkpoint đã seal bằng stage-model trước khi inference. Kaggle lưu output vào /kaggle/working; tải kết quả trước khi kết thúc phiên. Xem docs/run-gpu.md.


In [ ]:
from pathlib import Path
import os, sys, json, zipfile, stat, subprocess, runpy, time, hashlib

PLATFORM = "colab"
PLATFORM = os.environ.get("DATAMATE_PLATFORM", PLATFORM)  # "colab", "kaggle", "gpu"
MODEL_KEY = os.environ.get("DATAMATE_MODEL_KEY", "qwen35-9b")  # or "gemma4-12b"
OWNER = "hai"  # "thinh" or your name
RUN_NAME = time.strftime("%Y%m%d-%H%M%S")
RUN_LIMIT = None
IDS_FILE = None  # optional JSON array of selected question IDs
RESUME = False
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE = Path("/content/drive/MyDrive/DataMate")
    WORK = Path("/content/datamate")
    SOURCE_ZIP = STORAGE / "datamate-colab-source.zip"
    TRAIN_ROOT = STORAGE / "data/train"
    TRAIN_ZIP = STORAGE / "train.zip"
    MODEL_SOURCE_DIR = STORAGE / "models" / MODEL_KEY
elif PLATFORM == "kaggle":
    WORK = Path("/kaggle/working/datamate")
    STORAGE = Path("/kaggle/working/datamate-results")
    SOURCE_ZIP = Path("/kaggle/input/datamate-source/datamate-colab-source.zip")
    TRAIN_ROOT = Path("/kaggle/input/datamate-data/train")
    TRAIN_ZIP = None
    MODEL_SOURCE_DIR = Path("/kaggle/input/datamate-model") / MODEL_KEY
elif PLATFORM == "gpu":
    WORK = Path("/work/datamate")
    STORAGE = Path("/work/datamate-results")
    SOURCE_ZIP = Path("/work/inputs/datamate-colab-source.zip")
    TRAIN_ROOT = Path("/work/data/train")
    TRAIN_ZIP = None
    MODEL_SOURCE_DIR = Path("/work/models") / MODEL_KEY
else:
    raise ValueError("PLATFORM must be colab, kaggle or gpu")

WORK = Path(os.environ.get("DATAMATE_WORK_DIR", str(WORK)))
SOURCE_ZIP = Path(os.environ.get("DATAMATE_SOURCE_ZIP", str(SOURCE_ZIP)))
TRAIN_ROOT = Path(os.environ.get("DATAMATE_SPLIT_ROOT", str(TRAIN_ROOT)))
MODEL_SOURCE_DIR = Path(os.environ.get("DATAMATE_MODEL_DIR", str(MODEL_SOURCE_DIR)))

COPY_MODEL_TO_SCRATCH = PLATFORM == "colab"
SAVE = Path(os.environ.get("DATAMATE_RESULT_DIR", str(STORAGE / "runs" / OWNER / RUN_NAME)))
SAVE.mkdir(parents=True, exist_ok=True)
if SOURCE_ZIP.is_file():
    with SOURCE_ZIP.open("rb") as handle:
        source_hash = hashlib.file_digest(handle, "sha256").hexdigest()
    WORK = WORK / ("source-" + source_hash[:12])
    receipt = WORK / ".source_archive_sha256"
    if WORK.exists() and any(WORK.iterdir()) and (not receipt.is_file() or receipt.read_text().strip() != source_hash):
        raise RuntimeError("Source workspace is incomplete or differs from ZIP; choose a new WORK folder")
    WORK.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        for entry in archive.infolist():
            target = (WORK / entry.filename).resolve()
            if not target.is_relative_to(WORK.resolve()) or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Unsafe source ZIP path")
        archive.extractall(WORK)
    receipt.write_text(source_hash + "\n", encoding="utf-8")
elif not (WORK / "src/datamate/__init__.py").is_file():
    raise FileNotFoundError("Provide SOURCE_ZIP or an existing repository at WORK")
loaded = sys.modules.get("datamate")
if loaded and not Path(loaded.__file__).resolve().is_relative_to(WORK / "src"):
    raise RuntimeError("Another DataMate source is already loaded; restart the kernel")
CONFIG_FILE = Path(os.environ["DATAMATE_CONFIG"]) if os.environ.get("DATAMATE_CONFIG") else WORK / "configs" / (MODEL_KEY + ".json")
if not TRAIN_ROOT.is_dir():
    if TRAIN_ZIP is None or not TRAIN_ZIP.is_file():
        raise FileNotFoundError("Set TRAIN_ROOT to an extracted split containing questions.jsonl")
    helpers = runpy.run_path(str(WORK / "tools/colab_archives.py"))
    TRAIN_ROOT = helpers["prepare_train_archive"](TRAIN_ZIP, WORK, WORK / "configs/training_sources.json")
os.chdir(WORK)
sys.path.insert(0, str(WORK))
os.environ.update(HF_HOME=str(WORK / ".cache/huggingface"), TORCH_HOME=str(WORK / ".cache/torch"),
                  PIP_CACHE_DIR=str(WORK / ".cache/pip"))
print("Source:", WORK, "| data:", TRAIN_ROOT, "| results:", SAVE)


## Cài dependency GPU
Giữ PyTorch/CUDA đã có; xử lý lỗi môi trường trước khi đo baseline.


In [ ]:
import os
import torch
constraints = WORK / "colab-constraints.txt"
constraints.write_text("torch==" + torch.__version__ + "\n")
backend = os.environ.get("DATAMATE_BACKEND", "transformers")
install = [sys.executable, "-m", "pip", "install", "-e", ".[vllm]" if backend == "vllm" else ".[llm]"]
if backend != "vllm": install += ["-c", str(constraints)]
subprocess.run(install, cwd=WORK, check=True)
if backend == "vllm":
    from importlib.metadata import version
    if version("torch").split("+")[0] != torch.__version__.split("+")[0]:
        raise RuntimeError("Torch changed during vLLM install; restart kernel before continuing")

subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv"],check=True)
assert torch.cuda.is_available(), "Select a CUDA GPU runtime first"
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

import importlib
source_dir = (WORK / "src").resolve()
if not (source_dir / "datamate/__init__.py").is_file():
    raise RuntimeError("Gói source thiếu src/datamate. Thay source ZIP mới rồi chạy ô đầu.")
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))
importlib.invalidate_caches()
import datamate
print("DataMate import OK:", datamate.__version__, datamate.__file__)


## Chuẩn bị cấu hình và checkpoint local
Chưa load model; bước chạy sau tính thời gian từ trước load.


In [ ]:
from datamate.config import Settings
from pathlib import Path
import runpy

verify_runtime = runpy.run_path(str(WORK / "tools/colab_runtime_check.py"))["verify_runtime"]
print("Preflight:", verify_runtime(WORK))
if not (MODEL_SOURCE_DIR / "artifact_manifest.json").is_file():
    raise FileNotFoundError(
        "Prepare the pinned checkpoint first: python -m datamate stage-model --model "
        + MODEL_KEY + " --output " + str(MODEL_SOURCE_DIR)
    )
MODEL_DIR = MODEL_SOURCE_DIR
if COPY_MODEL_TO_SCRATCH:
    prepare_local_model = runpy.run_path(str(WORK / "tools/colab_model_cache.py"))["prepare_local_model"]
    MODEL_DIR = prepare_local_model(MODEL_SOURCE_DIR, WORK / ".cache/models", MODEL_KEY)
settings = Settings.load(CONFIG_FILE, model_key=MODEL_KEY, model_dir=str(MODEL_DIR),
                         split_root=str(TRAIN_ROOT), cache_dir=str(WORK / ".cache/datamate"),
                         result_dir=str(SAVE))
print("Prepared config; the next cell loads one model and runs the shared pipeline.")


## Chạy inference
Baseline trước rồi bản tích hợp trên cùng bộ đã chọn. Mỗi cấu hình dùng RUN_NAME mới; chỉ RESUME=True khi source/model/config không đổi. Log được ghi từng câu.


In [ ]:
from datamate.application import run_batch
from pathlib import Path
import json

selected_ids = set(json.loads(Path(IDS_FILE).read_text(encoding="utf-8"))) if IDS_FILE else None
summary = run_batch(settings, ids=selected_ids, limit=RUN_LIMIT, resume=RESUME)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Submission:", SAVE / "submission.jsonl", "| logs:", SAVE / "logs")
